<a href="https://colab.research.google.com/github/chrisanuo/chrisanuo/blob/main/Full_split_plot_ANOVA_bulk_soil_fractions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%reload_ext rpy2.ipython

In [8]:
%%R
# ==============================================================
# STEP 2: INSTALL AND LOAD REQUIRED PACKAGES
# ==============================================================

# We will use the 'pak' package which automatically resolves and installs
# missing system dependencies (like libgsl-dev or cmake) and uses fast parallel downloads.
if (!require("pak", character.only = TRUE, quietly = TRUE)) {
  install.packages("pak", repos = "https://cloud.r-project.org")
  library("pak")
}

packages <- c(
  "readxl",
  "dplyr",
  "tidyr",
  "purrr",
  "stringr",
  "janitor",
  "lme4",
  "lmerTest",
  "emmeans",
  "multcomp",
  "multcompView",
  "openxlsx"
)

# Install missing packages and their system dependencies automatically
pak::pkg_install(packages)

# Load packages
invisible(
  lapply(
    packages,
    library,
    character.only = TRUE
  )
)

# Use sum-to-zero contrasts for Type III ANOVA
options(
  contrasts = c("contr.sum", "contr.poly"),
  emmeans.lmer.df = "satterthwaite"
)

cat("All required packages were loaded successfully.\n")

Streaming output truncated to the last 5000 lines.
⸨████████▒▒   ⸩ | 📦  58/67 ⠼ 2 | ✅  44/67     | building dplyr, Matrix
⸨████████▒▒   ⸩ | 📦  58/67 ⠦ 2 | ✅  44/67     | building dplyr, Matrix
⸨████████▒▒   ⸩ | 📦  58/67 ⠇ 2 | ✅  44/67     | building dplyr, Matrix
⸨████████▒▒   ⸩ | 📦  58/67 ⠋ 2 | ✅  44/67     | building dplyr, Matrix
⸨████████▒▒   ⸩ | 📦  58/67 ⠹ 2 | ✅  44/67     | building dplyr, Matrix
⸨████████▒▒   ⸩ | 📦  58/67 ⠼ 2 | ✅  44/67     | building dplyr, Matrix
⸨████████▒▒   ⸩ | 📦  58/67 ⠦ 2 | ✅  44/67     | building dplyr, Matrix
⸨████████▒▒   ⸩ | 📦  58/67 ⠇ 2 | ✅  44/67     | building dplyr, Matrix
⸨████████▒▒   ⸩ | 📦  58/67 ⠋ 2 | ✅  44/67     | building dplyr, Matrix
⸨████████▒▒   ⸩ | 📦  58/67 ⠹ 2 | ✅  44/67     | building dplyr, Matrix
⸨████████▒▒   ⸩ | 📦  58/67 ⠼ 2 | ✅  44/67     | building dplyr, Matrix
⸨████████▒▒   ⸩ | 📦  58/67 ⠦ 2 | ✅  44/67     | building dplyr, Matrix
⸨████████▒▒   ⸩ | 📦  58/67 ⠇ 2 | ✅  44/67     | building dplyr, Matrix
⸨████████▒▒   ⸩ | 📦  58/67

Loading required package: Matrix

Attaching package: ‘Matrix’

The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack


Attaching package: ‘lmerTest’

The following object is masked from ‘package:lme4’:

    lmer

The following object is masked from ‘package:stats’:

    step

Welcome to emmeans.
Caution: You lose important information if you filter this package's results.
See '? untidy'
Loading required package: mvtnorm
Loading required package: survival
Loading required package: TH.data
Loading required package: MASS

Attaching package: ‘MASS’

The following object is masked from ‘package:dplyr’:

    select


Attaching package: ‘TH.data’

The following object is masked from ‘package:MASS’:

    geyser



In [10]:
%%R
# ==============================================================
# STEP 3A: LIST FILES UPLOADED TO COLAB
# ==============================================================

list.files("/content")

[1] "sample_data"


In [13]:
%%R
# ==============================================================
# STEP 3B: DEFINE INPUT AND OUTPUT FILES
# ==============================================================

# Replace the filename with the exact name of your uploaded workbook
input_file <- "/content/soil_fraction_raw_data.xlsx"

# Name of the Excel workbook that will contain the results
output_file <- "/content/Complete_Split_Plot_Analysis.xlsx"

# Fisher's LSD significance level
alpha_level <- 0.05

# Confirm that the input file exists
if (!file.exists(input_file)) {
  stop(
    paste0(
      "The workbook was not found at:\n",
      input_file,
      "\n\nCheck the filename using list.files('/content')."
    )
  )
}

cat("Input workbook found:\n", input_file, "\n")

Input workbook found:
 /content/soil_fraction_raw_data.xlsx 


In [18]:
%%R
input_file <- "/content/soil_fraction_raw_data.xlsx"

In [19]:
%%R
# ==============================================================
# STEP 4: DETECT WORKSHEETS
# ==============================================================

sheet_names <- excel_sheets(input_file)

cat("Worksheets detected in the workbook:\n")
print(sheet_names)

# Confirm that a 75-100 cm worksheet is present
deepest_sheet_present <- any(
  str_detect(
    sheet_names,
    regex("75\\s*[-–_]\\s*100", ignore_case = TRUE)
  )
)

if (!deepest_sheet_present) {
  warning(
    paste0(
      "No worksheet clearly corresponding to 75-100 cm was detected.\n",
      "Check the spelling of the worksheet name."
    )
  )
} else {
  cat("\nThe 75-100 cm worksheet was detected.\n")
}

Worksheets detected in the workbook:
[1] "0-5"    "5-15"   "15-30"  "30-50"  "50-75"  "75-100"

The 75-100 cm worksheet was detected.


In [22]:
%%R
# ==============================================================
# STEP 5: INSPECT COLUMN NAMES
# ==============================================================

for (sheet in sheet_names) {

  temporary_data <- read_excel(
    input_file,
    sheet = sheet
  ) |>
    janitor::clean_names()

  cat("\n---------------------------------------------\n")
  cat("Worksheet:", sheet, "\n")
  cat("---------------------------------------------\n")

  print(names(temporary_data))
}


---------------------------------------------
Worksheet: 0-5 
---------------------------------------------
 [1] "rep"                      "treatments"              
 [3] "crop_rotation"            "depth"                   
 [5] "actual_depth"             "bulk_c_g_kg"             
 [7] "bulk_n_g_kg"              "pom_c_g_kg"              
 [9] "pom_n_g_kg"               "maom_c_g_kg"             
[11] "maom_n_g_kg"              "total_c_recovered_g_kg"  
[13] "total_n_recovered_g_kg"   "pom_c_recovered_percent" 
[15] "pom_n_recovered_percent"  "maom_c_recovered_percent"
[17] "maom_n_recovered_percent" "total_c_percent"         
[19] "total_n_percent"          "bulk_c_n"                
[21] "pom_c_n"                  "maom_c_n"                

---------------------------------------------
Worksheet: 5-15 
---------------------------------------------
 [1] "rep"                      "treatments"              
 [3] "crop_rotation"            "depth"                   
 [5] "actual_d

In [24]:
%%R
# ==============================================================
# STEP 6: COLUMN IDENTIFICATION FUNCTIONS
# ==============================================================

find_column <- function(
  data,
  possible_names,
  required = TRUE
) {

  available_names <- names(data)

  matched_name <- intersect(
    possible_names,
    available_names
  )

  if (length(matched_name) > 0) {
    return(matched_name[1])
  }

  if (required) {
    stop(
      paste0(
        "\nA required column could not be identified.\n",
        "Expected one of these names:\n",
        paste(possible_names, collapse = ", "),
        "\n\nColumns available in the worksheet:\n",
        paste(available_names, collapse = ", ")
      )
    )
  }

  return(NA_character_)
}


rename_if_found <- function(
  data,
  old_name,
  new_name
) {

  if (
    !is.na(old_name) &&
    old_name %in% names(data)
  ) {
    names(data)[names(data) == old_name] <- new_name
  }

  return(data)
}

cat("Column-identification functions created.\n")

Column-identification functions created.


In [28]:
%%R
# ==============================================================
# STEP 7: READ AND STANDARDIZE EACH DEPTH WORKSHEET
# ==============================================================

read_depth_sheet <- function(sheet) {

  raw_data <- read_excel(
    input_file,
    sheet = sheet
  ) |>
    janitor::clean_names()

  cat("\nReading worksheet:", sheet, "\n")

  # ------------------------------------------------------------
  # Identify experimental-design columns
  # ------------------------------------------------------------

  rep_col <- find_column(
    raw_data,
    c(
      "rep",
      "replicate",
      "replication",
      "block",
      "block_number"
    )
  )

  tillage_col <- find_column(
    raw_data,
    c(
      "tillage",
      "treatments",
      "treatment",
      "tillage_treatment",
      "tillage_system"
    )
  )

  rotation_col <- find_column(
    raw_data,
    c(
      "rotation",
      "crop_rotation",
      "crop_rotations",
      "cropping_system",
      "crop"
    )
  )

  # ------------------------------------------------------------
  # Identify measured response variables
  # ------------------------------------------------------------

  bulk_oc_col <- find_column(
    raw_data,
    c(
      "bulk_c_g_kg",
      "bulk_oc",
      "bulk_soil_oc",
      "soil_oc",
      "soc",
      "organic_carbon",
      "bulk_c",
      "total_organic_carbon"
    )
  )

  bulk_n_col <- find_column(
    raw_data,
    c(
      "bulk_n_g_kg",
      "bulk_n",
      "bulk_nt",
      "bulk_soil_n",
      "bulk_soil_nt",
      "soil_n",
      "soil_nt",
      "nt",
      "total_n",
      "total_nitrogen"
    )
  )

  pom_oc_col <- find_column(
    raw_data,
    c(
      "pom_c_g_kg",
      "pom_oc",
      "pom_c",
      "poc",
      "particulate_organic_matter_oc",
      "particulate_organic_matter_c",
      "pom_organic_carbon"
    )
  )

  pom_n_col <- find_column(
    raw_data,
    c(
      "pom_n_g_kg",
      "pom_n",
      "pom_nt",
      "particulate_organic_matter_n",
      "particulate_organic_matter_nt",
      "pom_nitrogen"
    )
  )

  maom_oc_col <- find_column(
    raw_data,
    c(
      "maom_c_g_kg",
      "maom_oc",
      "maom_c",
      "mineral_associated_organic_matter_oc",
      "mineral_associated_organic_matter_c",
      "maom_organic_carbon"
    )
  )

  maom_n_col <- find_column(
    raw_data,
    c(
      "maom_n_g_kg",
      "maom_n",
      "maom_nt",
      "mineral_associated_organic_matter_n",
      "mineral_associated_organic_matter_nt",
      "maom_nitrogen"
    )
  )

  # ------------------------------------------------------------
  # Rename columns consistently
  # ------------------------------------------------------------

  standardized_data <- raw_data |>
    rename_if_found(rep_col, "Rep") |>
    rename_if_found(tillage_col, "Tillage") |>
    rename_if_found(rotation_col, "Rotation") |>
    rename_if_found(bulk_oc_col, "Bulk_OC") |>
    rename_if_found(bulk_n_col, "Bulk_N") |>
    rename_if_found(pom_oc_col, "POM_OC") |>
    rename_if_found(pom_n_col, "POM_N") |>
    rename_if_found(maom_oc_col, "MAOM_OC") |>
    rename_if_found(maom_n_col, "MAOM_N")

  # ------------------------------------------------------------
  # Convert variables to the correct data type
  # ------------------------------------------------------------

  standardized_data <- standardized_data |>
    mutate(
      Depth = sheet,

      Rep = factor(Rep),
      Rotation = factor(Rotation),
      Tillage = factor(Tillage),

      Bulk_OC = as.numeric(Bulk_OC),
      Bulk_N = as.numeric(Bulk_N),
      POM_OC = as.numeric(POM_OC),
      POM_N = as.numeric(POM_N),
      MAOM_OC = as.numeric(MAOM_OC),
      MAOM_N = as.numeric(MAOM_N)
    )

  # ------------------------------------------------------------
  # Calculate C/N ratios
  #
  # A ratio is returned as missing when N is missing or equal to 0.
  # MAOM C/N is calculated for every worksheet, including 75-100 cm.
  # ------------------------------------------------------------

  standardized_data <- standardized_data |>
    mutate(
      Bulk_CN = case_when(
        !is.na(Bulk_OC) &
          !is.na(Bulk_N) &
          Bulk_N > 0 ~ Bulk_OC / Bulk_N,
        TRUE ~ NA_real_
      ),

      POM_CN = case_when(
        !is.na(POM_OC) &
          !is.na(POM_N) &
          POM_N > 0 ~ POM_OC / POM_N,
        TRUE ~ NA_real_
      ),

      MAOM_CN = case_when(
        !is.na(MAOM_OC) &
          !is.na(MAOM_N) &
          MAOM_N > 0 ~ MAOM_OC / MAOM_N,
        TRUE ~ NA_real_
      )
    )

  # Keep only variables needed for the analysis
  standardized_data |>
    dplyr::select(
      Depth,
      Rep,
      Rotation,
      Tillage,
      Bulk_OC,
      Bulk_N,
      POM_OC,
      POM_N,
      MAOM_OC,
      MAOM_N,
      Bulk_CN,
      POM_CN,
      MAOM_CN
    )
}


# Read every worksheet and combine them into one dataset
all_data <- map_dfr(
  sheet_names,
  read_depth_sheet
)

cat("\nData from all worksheets were combined successfully.\n")
cat("Number of rows:", nrow(all_data), "\n")
cat("Number of columns:", ncol(all_data), "\n")


Reading worksheet: 0-5 

Reading worksheet: 5-15 

Reading worksheet: 15-30 

Reading worksheet: 30-50 

Reading worksheet: 50-75 

Reading worksheet: 75-100 

Data from all worksheets were combined successfully.
Number of rows: 216 
Number of columns: 13 


In [30]:
%%R
# ==============================================================
# STEP 8: STANDARDIZE AND ORDER DEPTHS
# ==============================================================

standardize_depth <- function(x) {

  x <- as.character(x)

  x <- str_replace_all(
    x,
    "[–—_]",
    "-"
  )

  x <- str_replace_all(
    x,
    "\\s+",
    ""
  )

  x <- str_replace_all(
    x,
    "cm",
    ""
  )

  return(x)
}

all_data <- all_data |>
  mutate(
    Depth = standardize_depth(Depth)
  )

desired_depth_order <- c(
  "0-5",
  "5-15",
  "15-30",
  "30-50",
  "50-75",
  "75-100"
)

unrecognized_depths <- setdiff(
  unique(all_data$Depth),
  desired_depth_order
)

if (length(unrecognized_depths) > 0) {

  warning(
    paste0(
      "The following worksheet names were not recognized as standard depths:\n",
      paste(unrecognized_depths, collapse = ", ")
    )
  )
}

actual_depth_order <- c(
  desired_depth_order[
    desired_depth_order %in% unique(all_data$Depth)
  ],
  unrecognized_depths
)

all_data <- all_data |>
  mutate(
    Depth = factor(
      Depth,
      levels = actual_depth_order
    )
  )

cat("Depths included in the dataset:\n")
print(levels(all_data$Depth))

cat("\nTillage levels:\n")
print(levels(all_data$Tillage))

cat("\nCrop-rotation levels:\n")
print(levels(all_data$Rotation))

Depths included in the dataset:
[1] "0-5"    "5-15"   "15-30"  "30-50"  "50-75"  "75-100"

Tillage levels:
[1] "Chisel plow" "MB plow"     "No-till"    

Crop-rotation levels:
[1] "B-B" "C-B" "C-C"


In [33]:
%%R
# ==============================================================
# STEP 9: CHECK MAOM C/N AT 75-100 CM
# ==============================================================

maom_cn_75_100_data <- all_data |>
  filter(
    Depth == "75-100"
  ) |>
  dplyr::select(
    Depth,
    Rep,
    Rotation,
    Tillage,
    MAOM_OC,
    MAOM_N,
    MAOM_CN
  )

cat("MAOM C/N observations at 75-100 cm:\n")
print(maom_cn_75_100_data)

cat(
  "\nNumber of non-missing MAOM C/N observations at 75-100 cm:",
  sum(!is.na(maom_cn_75_100_data$MAOM_CN)),
  "\n"
)

if (
  nrow(maom_cn_75_100_data) == 0
) {
  stop(
    "No observations were found for the 75-100 cm depth."
  )
}

if (
  sum(!is.na(maom_cn_75_100_data$MAOM_CN)) == 0
) {
  stop(
    paste0(
      "MAOM C/N could not be calculated at 75-100 cm.\n",
      "Check the MAOM OC and MAOM N columns in that worksheet."
    )
  )
}

MAOM C/N observations at 75-100 cm:
# A tibble: 36 × 7
   Depth  Rep   Rotation Tillage MAOM_OC MAOM_N MAOM_CN
   <fct>  <fct> <fct>    <fct>     <dbl>  <dbl>   <dbl>
 1 75-100 1     B-B      No-till    1.20  0.119   10.1 
 2 75-100 2     B-B      No-till    1.53  0.107   14.3 
 3 75-100 3     B-B      No-till    1.56  0.125   12.5 
 4 75-100 4     B-B      No-till    1.44  0.140   10.3 
 5 75-100 1     C-B      No-till    1.84  0.180   10.3 
 6 75-100 2     C-B      No-till    1.38  0.152    9.09
 7 75-100 3     C-B      No-till    1.80  0.147   12.3 
 8 75-100 4     C-B      No-till    1.65  0.154   10.7 
 9 75-100 1     C-C      No-till    2.11  0.125   16.9 
10 75-100 2     C-C      No-till    2.03  0.120   16.8 
# ℹ 26 more rows
# ℹ Use `print(n = ...)` to see more rows

Number of non-missing MAOM C/N observations at 75-100 cm: 36 


In [42]:
%%R
# ==============================================================
# STEP 10: DEFINE RESPONSE VARIABLES AND OUTPUT FUNCTIONS
# ==============================================================

response_variables <- c(
  "Bulk_OC",
  "Bulk_N",
  "POM_OC",
  "POM_N",
  "MAOM_OC",
  "MAOM_N",
  "Bulk_CN",
  "POM_CN",
  "MAOM_CN"
)

response_labels <- c(
  Bulk_OC = "Bulk soil OC",
  Bulk_N = "Bulk soil Nt",
  POM_OC = "POM-OC",
  POM_N = "POM-N",
  MAOM_OC = "MAOM-OC",
  MAOM_N = "MAOM-N",
  Bulk_CN = "Bulk soil C/N",
  POM_CN = "POM C/N",
  MAOM_CN = "MAOM C/N"
)


clean_cld <- function(cld_table) {

  cld_table |>
    as.data.frame() |>
    mutate(
      .group = str_squish(.group)
    )
}


extract_anova <- function(
  model,
  variable,
  depth
) {

  result <- anova(
    model,
    type = 3,
    ddf = "Satterthwaite"
  ) |>
    as.data.frame()

  result$Effect <- rownames(result)
  rownames(result) <- NULL

  names(result) <- names(result) |>
    str_replace("NumDF", "Numerator_df") |>
    str_replace("DenDF", "Denominator_df") |>
    str_replace("F value", "F_value") |>
    str_replace("Pr\\(>F\\)", "p_value")

  result |>
    mutate(
      Variable = response_labels[[variable]],
      Variable_code = variable,
      Depth = as.character(depth),

      Significance = case_when(
        p_value <= 0.001 ~ "***",
        p_value <= 0.01 ~ "**",
        p_value <= 0.05 ~ "*",
        TRUE ~ "ns"
      )
    ) |>
    dplyr::select(
      Variable,
      Variable_code,
      Depth,
      Effect,
      everything()
    )
}


extract_diagnostics <- function(
  model,
  variable,
  depth,
  n_used
) {

  model_residuals <- residuals(model)

  shapiro_result <- tryCatch(
    shapiro.test(model_residuals),
    error = function(e) NULL
  )

  convergence_messages <- model@optinfo$conv$lme4$messages

  if (is.null(convergence_messages)) {
    convergence_messages <- ""
  } else {
    convergence_messages <- paste(
      convergence_messages,
      collapse = "; "
    )
  }

  data.frame(
    Variable = response_labels[[variable]],
    Variable_code = variable,
    Depth = as.character(depth),
    N_used = n_used,
    Singular_fit = isSingular(model, tol = 1e-4),
    Residual_SD = sigma(model),

    Shapiro_W = if (
      is.null(shapiro_result)
    ) {
      NA_real_
    } else {
      unname(shapiro_result$statistic)
    },

    Shapiro_p = if (
      is.null(shapiro_result)
    ) {
      NA_real_
    } else {
      shapiro_result$p.value
    },

    Convergence_message = convergence_messages,

    stringsAsFactors = FALSE
  )
}

cat("Response variables and extraction functions created.\n")

Response variables and extraction functions created.


In [37]:
%%R
# This is a random effect structure for lme4. It should be used inside a model formula.
# Example: lmer(Response ~ Treatment + (1 | Rep) + (1 | Rep:Rotation), data = my_data)

# (1 | Rep) + (1 | Rep:Rotation)


NULL


In [43]:
%%R
# ==============================================================
# STEP 11: RUN THE COMPLETE SPLIT-PLOT ANALYSIS
# ==============================================================

# Create empty lists for storing results
anova_results <- list()
tillage_emmeans_results <- list()
rotation_emmeans_results <- list()
interaction_emmeans_results <- list()
tillage_within_rotation_results <- list()
rotation_within_tillage_results <- list()
diagnostic_results <- list()
analysis_log <- list()

result_counter <- 1


for (variable in response_variables) {

  for (depth_level in levels(all_data$Depth)) {

    cat(
      "\n====================================================\n",
      "Analyzing:", response_labels[[variable]],
      "\nDepth:", depth_level,
      "\n====================================================\n"
    )

    # Select complete observations for the current variable and depth
    analysis_data <- all_data |>
      filter(
        Depth == depth_level
      ) |>
      dplyr::select(
        Rep,
        Rotation,
        Tillage,
        all_of(variable)
      ) |>
      rename(
        Response = all_of(variable)
      ) |>
      tidyr::drop_na() |>
      droplevels()

    n_used <- nrow(analysis_data)

    # Check that sufficient data and factor levels are available
    if (
      n_used < 6 ||
      nlevels(analysis_data$Rep) < 2 ||
      nlevels(analysis_data$Rotation) < 2 ||
      nlevels(analysis_data$Tillage) < 2
    ) {

      warning_message <- paste(
        "Analysis skipped:",
        response_labels[[variable]],
        "at",
        depth_level,
        "because insufficient complete observations or factor levels were available."
      )

      warning(warning_message)

      analysis_log[[result_counter]] <- data.frame(
        Variable = response_labels[[variable]],
        Variable_code = variable,
        Depth = depth_level,
        N_used = n_used,
        Status = "Skipped",
        Message = warning_message
      )

      result_counter <- result_counter + 1
      next
    }

    # Fit the split-plot mixed model
    model <- tryCatch(

      lmer(
        Response ~ Rotation * Tillage +
          (1 | Rep) +
          (1 | Rep:Rotation),
        data = analysis_data,
        REML = TRUE
      ),

      error = function(e) e
    )

    # Record failed models
    if (inherits(model, "error")) {

      error_message <- paste(
        "Model failed:",
        response_labels[[variable]],
        "at",
        depth_level,
        "-",
        model$message
      )

      warning(error_message)

      analysis_log[[result_counter]] <- data.frame(
        Variable = response_labels[[variable]],
        Variable_code = variable,
        Depth = depth_level,
        N_used = n_used,
        Status = "Failed",
        Message = error_message
      )

      result_counter <- result_counter + 1
      next
    }

    # ----------------------------------------------------------
    # Type III ANOVA
    # ----------------------------------------------------------

    anova_results[[result_counter]] <- extract_anova(
      model = model,
      variable = variable,
      depth = depth_level
    )

    # ----------------------------------------------------------
    # Main effect of tillage averaged across crop rotations
    # ----------------------------------------------------------

    tillage_emm <- emmeans(
      model,
      ~ Tillage,
      lmer.df = "satterthwaite"
    )

    tillage_cld <- multcomp::cld(
      tillage_emm,
      alpha = alpha_level,
      adjust = "none",
      Letters = letters,
      sort = FALSE
    ) |>
      clean_cld() |>
      mutate(
        Variable = response_labels[[variable]],
        Variable_code = variable,
        Depth = depth_level,
        Effect = "Tillage main effect"
      ) |>
      dplyr::select(
        Variable,
        Variable_code,
        Depth,
        Effect,
        everything()
      )

    tillage_emmeans_results[[result_counter]] <- tillage_cld

    # ----------------------------------------------------------
    # Main effect of crop rotation averaged across tillage
    # ----------------------------------------------------------

    rotation_emm <- emmeans(
      model,
      ~ Rotation,
      lmer.df = "satterthwaite"
    )

    rotation_cld <- multcomp::cld(
      rotation_emm,
      alpha = alpha_level,
      adjust = "none",
      Letters = letters,
      sort = FALSE
    ) |>
      clean_cld() |>
      mutate(
        Variable = response_labels[[variable]],
        Variable_code = variable,
        Depth = depth_level,
        Effect = "Crop rotation main effect"
      ) |>
      dplyr::select(
        Variable,
        Variable_code,
        Depth,
        Effect,
        everything()
      )

    rotation_emmeans_results[[result_counter]] <- rotation_cld

    # ----------------------------------------------------------
    # All tillage × rotation combinations
    # ----------------------------------------------------------

    interaction_emm <- emmeans(
      model,
      ~ Rotation * Tillage,
      lmer.df = "satterthwaite"
    )

    interaction_cld <- multcomp::cld(
      interaction_emm,
      alpha = alpha_level,
      adjust = "none",
      Letters = letters,
      sort = FALSE
    ) |>
      clean_cld() |>
      mutate(
        Variable = response_labels[[variable]],
        Variable_code = variable,
        Depth = depth_level,
        Effect = "Rotation × Tillage combinations"
      ) |>
      dplyr::select(
        Variable,
        Variable_code,
        Depth,
        Effect,
        everything()
      )

    interaction_emmeans_results[[result_counter]] <- interaction_cld

    # ----------------------------------------------------------
    # Simple effect: tillage within each crop rotation
    # ----------------------------------------------------------

    tillage_within_rotation_emm <- emmeans(
      model,
      ~ Tillage | Rotation,
      lmer.df = "satterthwaite"
    )

    tillage_within_rotation_cld <- multcomp::cld(
      tillage_within_rotation_emm,
      alpha = alpha_level,
      adjust = "none",
      Letters = letters,
      sort = FALSE
    ) |>
      clean_cld() |>
      mutate(
        Variable = response_labels[[variable]],
        Variable_code = variable,
        Depth = depth_level,
        Effect = "Tillage within crop rotation"
      ) |>
      dplyr::select(
        Variable,
        Variable_code,
        Depth,
        Effect,
        everything()
      )

    tillage_within_rotation_results[[result_counter]] <-
      tillage_within_rotation_cld

    # ----------------------------------------------------------
    # Simple effect: crop rotation within each tillage system
    # ----------------------------------------------------------

    rotation_within_tillage_emm <- emmeans(
      model,
      ~ Rotation | Tillage,
      lmer.df = "satterthwaite"
    )

    rotation_within_tillage_cld <- multcomp::cld(
      rotation_within_tillage_emm,
      alpha = alpha_level,
      adjust = "none",
      Letters = letters,
      sort = FALSE
    ) |>
      clean_cld() |>
      mutate(
        Variable = response_labels[[variable]],
        Variable_code = variable,
        Depth = depth_level,
        Effect = "Crop rotation within tillage"
      ) |>
      dplyr::select(
        Variable,
        Variable_code,
        Depth,
        Effect,
        everything()
      )

    rotation_within_tillage_results[[result_counter]] <-
      rotation_within_tillage_cld

    # ----------------------------------------------------------
    # Model diagnostics
    # ----------------------------------------------------------

    diagnostic_results[[result_counter]] <- extract_diagnostics(
      model = model,
      variable = variable,
      depth = depth_level,
      n_used = n_used
    )

    # Record successful model
    analysis_log[[result_counter]] <- data.frame(
      Variable = response_labels[[variable]],
      Variable_code = variable,
      Depth = depth_level,
      N_used = n_used,
      Status = "Completed",
      Message = ""
    )

    result_counter <- result_counter + 1
  }
}


cat("\nAll model-fitting loops have finished.\n")


 Analyzing: Bulk soil OC 
Depth: 0-5 

 Analyzing: Bulk soil OC 
Depth: 5-15 

 Analyzing: Bulk soil OC 
Depth: 15-30 

 Analyzing: Bulk soil OC 
Depth: 30-50 

 Analyzing: Bulk soil OC 
Depth: 50-75 

 Analyzing: Bulk soil OC 
Depth: 75-100 

 Analyzing: Bulk soil Nt 
Depth: 0-5 

 Analyzing: Bulk soil Nt 
Depth: 5-15 

 Analyzing: Bulk soil Nt 
Depth: 15-30 

 Analyzing: Bulk soil Nt 
Depth: 30-50 

 Analyzing: Bulk soil Nt 
Depth: 50-75 

 Analyzing: Bulk soil Nt 
Depth: 75-100 

 Analyzing: POM-OC 
Depth: 0-5 

 Analyzing: POM-OC 
Depth: 5-15 

 Analyzing: POM-OC 
Depth: 15-30 

 Analyzing: POM-OC 
Depth: 30-50 

 Analyzing: POM-OC 
Depth: 50-75 

 Analyzing: POM-OC 
Depth: 75-100 

 Analyzing: POM-N 
Depth: 0-5 

 Analyzing: POM-N 
Depth: 5-15 

 Analyzing: POM-N 
Depth: 15-30 

 Analyzing: POM-N 
Depth: 30-50 

 Analyzing: POM-N 
Depth: 50-75 

 Analyzing: POM-N 
Depth: 75-100 

 Analyzing: MAOM-OC 
Depth: 0-5 

 Analyzing: MAOM-OC 
Depth: 5-15 

 Analyzing: MAOM-OC 
Depth: 15-3

boundary (singular) fit: see help('isSingular')
NOTE: Results may be misleading due to involvement in interactions
NOTE: Results may be misleading due to involvement in interactions
NOTE: Results may be misleading due to involvement in interactions
NOTE: Results may be misleading due to involvement in interactions
boundary (singular) fit: see help('isSingular')
NOTE: Results may be misleading due to involvement in interactions
NOTE: Results may be misleading due to involvement in interactions
boundary (singular) fit: see help('isSingular')
NOTE: Results may be misleading due to involvement in interactions
NOTE: Results may be misleading due to involvement in interactions
boundary (singular) fit: see help('isSingular')
NOTE: Results may be misleading due to involvement in interactions
NOTE: Results may be misleading due to involvement in interactions
boundary (singular) fit: see help('isSingular')
NOTE: Results may be misleading due to involvement in interactions
NOTE: Results may be mi

In [45]:
%%R
# ==============================================================
# STEP 12: COMBINE ALL ANALYSIS RESULTS
# ==============================================================

anova_table <- bind_rows(anova_results)

tillage_table <- bind_rows(
  tillage_emmeans_results
)

rotation_table <- bind_rows(
  rotation_emmeans_results
)

interaction_table <- bind_rows(
  interaction_emmeans_results
)

tillage_within_rotation_table <- bind_rows(
  tillage_within_rotation_results
)

rotation_within_tillage_table <- bind_rows(
  rotation_within_tillage_results
)

diagnostics_table <- bind_rows(
  diagnostic_results
)

analysis_log_table <- bind_rows(
  analysis_log
)

cat("Number of completed analyses:\n")

print(
  analysis_log_table |>
    count(Status)
)

cat("\nFirst rows of the Type III ANOVA results:\n")
print(head(anova_table))

Number of completed analyses:
     Status  n
1 Completed 54

First rows of the Type III ANOVA results:
      Variable Variable_code Depth           Effect     Sum Sq     Mean Sq
1 Bulk soil OC       Bulk_OC   0-5         Rotation  104.09607   52.048035
2 Bulk soil OC       Bulk_OC   0-5          Tillage 2097.36809 1048.684045
3 Bulk soil OC       Bulk_OC   0-5 Rotation:Tillage   50.48413   12.621033
4 Bulk soil OC       Bulk_OC  5-15         Rotation   23.58693   11.793465
5 Bulk soil OC       Bulk_OC  5-15          Tillage   62.67337   31.336687
6 Bulk soil OC       Bulk_OC  5-15 Rotation:Tillage   16.20926    4.052315
  Numerator_df Denominator_df    F_value      p_value Significance
1            2      24.000000   9.523504 9.020262e-04          ***
2            2      24.000000 191.883270 1.728207e-15          ***
3            4      24.000000   2.309337 8.709884e-02           ns
4            2       6.000005   4.096346 7.555430e-02           ns
5            2      17.999991  10.884

In [48]:
%%R
# ==============================================================
# STEP 13: CREATE COMPACT ANOVA SUMMARY
# ==============================================================

compact_anova <- anova_table |>
  filter(
    Effect %in% c(
      "Rotation",
      "Tillage",
      "Rotation:Tillage",
      "Tillage:Rotation"
    )
  ) |>
  mutate(
    Effect = recode(
      Effect,
      "Rotation" = "Crop rotation",
      "Tillage" = "Tillage",
      "Rotation:Tillage" = "Tillage × Rotation",
      "Tillage:Rotation" = "Tillage × Rotation"
    ),

    Variable_order = match(
      Variable_code,
      response_variables
    ),

    Depth_order = match(
      Depth,
      desired_depth_order
    ),

    Effect_order = match(
      Effect,
      c(
        "Tillage",
        "Crop rotation",
        "Tillage × Rotation"
      )
    )
  ) |>
  arrange(
    Variable_order,
    Depth_order,
    Effect_order
  ) |>
  dplyr::select(
    Variable,
    Variable_code,
    Depth,
    Effect,
    Numerator_df,
    Denominator_df,
    F_value,
    p_value,
    Significance
  )

print(compact_anova)

         Variable Variable_code  Depth             Effect Numerator_df
1    Bulk soil OC       Bulk_OC    0-5            Tillage            2
2    Bulk soil OC       Bulk_OC    0-5      Crop rotation            2
3    Bulk soil OC       Bulk_OC    0-5 Tillage × Rotation            4
4    Bulk soil OC       Bulk_OC   5-15            Tillage            2
5    Bulk soil OC       Bulk_OC   5-15      Crop rotation            2
6    Bulk soil OC       Bulk_OC   5-15 Tillage × Rotation            4
7    Bulk soil OC       Bulk_OC  15-30            Tillage            2
8    Bulk soil OC       Bulk_OC  15-30      Crop rotation            2
9    Bulk soil OC       Bulk_OC  15-30 Tillage × Rotation            4
10   Bulk soil OC       Bulk_OC  30-50            Tillage            2
11   Bulk soil OC       Bulk_OC  30-50      Crop rotation            2
12   Bulk soil OC       Bulk_OC  30-50 Tillage × Rotation            4
13   Bulk soil OC       Bulk_OC  50-75            Tillage            2
14   B

In [50]:
%%R
# ==============================================================
# STEP 14: CREATE INTERPRETATION GUIDE
# ==============================================================

interaction_decisions <- compact_anova |>
  filter(
    Effect == "Tillage × Rotation"
  ) |>
  transmute(
    Variable,
    Variable_code,
    Depth,
    Interaction_p = p_value,

    Recommended_interpretation = case_when(

      Interaction_p <= alpha_level ~
        paste0(
          "Interaction significant: interpret tillage within ",
          "rotation and/or rotation within tillage."
        ),

      Interaction_p > alpha_level ~
        paste0(
          "Interaction not significant: interpret tillage and ",
          "crop rotation main effects."
        ),

      TRUE ~ "Interaction result unavailable."
    )
  )

print(interaction_decisions)

        Variable Variable_code  Depth Interaction_p
1   Bulk soil OC       Bulk_OC    0-5   0.087098841
2   Bulk soil OC       Bulk_OC   5-15   0.271511321
3   Bulk soil OC       Bulk_OC  15-30   0.600072083
4   Bulk soil OC       Bulk_OC  30-50   0.228660888
5   Bulk soil OC       Bulk_OC  50-75   0.585529730
6   Bulk soil OC       Bulk_OC 75-100   0.683319133
7   Bulk soil Nt        Bulk_N    0-5   0.124914262
8   Bulk soil Nt        Bulk_N   5-15   0.558306230
9   Bulk soil Nt        Bulk_N  15-30   0.218361335
10  Bulk soil Nt        Bulk_N  30-50   0.020564042
11  Bulk soil Nt        Bulk_N  50-75   0.567193330
12  Bulk soil Nt        Bulk_N 75-100   0.648906957
13        POM-OC        POM_OC    0-5   0.002143187
14        POM-OC        POM_OC   5-15   0.345554458
15        POM-OC        POM_OC  15-30   0.177400190
16        POM-OC        POM_OC  30-50   0.325077477
17        POM-OC        POM_OC  50-75   0.711607893
18        POM-OC        POM_OC 75-100   0.523091713
19         P

In [52]:
%%R
# ==============================================================
# STEP 15: VERIFY MAOM C/N ANALYSIS AT 75-100 CM
# ==============================================================

maom_cn_check <- analysis_log_table |>
  filter(
    Variable_code == "MAOM_CN",
    Depth == "75-100"
  )

cat("Analysis-log check:\n")
print(maom_cn_check)


maom_cn_anova_check <- compact_anova |>
  filter(
    Variable_code == "MAOM_CN",
    Depth == "75-100"
  )

cat("\nANOVA results for MAOM C/N at 75-100 cm:\n")
print(maom_cn_anova_check)


if (
  nrow(maom_cn_check) > 0 &&
  any(maom_cn_check$Status == "Completed")
) {

  cat(
    "\nSUCCESS: MAOM C/N at 75-100 cm was analyzed.\n"
  )

} else {

  warning(
    paste0(
      "MAOM C/N at 75-100 cm was not completed.\n",
      "Review the analysis log and the source data."
    )
  )
}

Analysis-log check:
  Variable Variable_code  Depth N_used    Status Message
1 MAOM C/N       MAOM_CN 75-100     36 Completed        

ANOVA results for MAOM C/N at 75-100 cm:
  Variable Variable_code  Depth             Effect Numerator_df Denominator_df
1 MAOM C/N       MAOM_CN 75-100            Tillage            2             27
2 MAOM C/N       MAOM_CN 75-100      Crop rotation            2             27
3 MAOM C/N       MAOM_CN 75-100 Tillage × Rotation            4             27
    F_value   p_value Significance
1 0.7962403 0.4613293           ns
2 2.0856533 0.1437865           ns
3 2.0385424 0.1171708           ns

SUCCESS: MAOM C/N at 75-100 cm was analyzed.


In [54]:
%%R
# ==============================================================
# STEP 16: EXPORT RESULTS TO EXCEL
# ==============================================================

workbook <- createWorkbook()


# --------------------------------------------------------------
# Define Excel styles
# --------------------------------------------------------------

header_style <- createStyle(
  textDecoration = "bold",
  halign = "center",
  valign = "center",
  border = "Bottom",
  wrapText = TRUE
)


write_formatted_sheet <- function(
  workbook,
  sheet_name,
  data
) {

  addWorksheet(
    workbook,
    sheet_name
  )

  writeData(
    workbook,
    sheet = sheet_name,
    x = data,
    headerStyle = header_style,
    withFilter = TRUE
  )

  freezePane(
    workbook,
    sheet = sheet_name,
    firstRow = TRUE
  )

  if (ncol(data) > 0) {

    setColWidths(
      workbook,
      sheet = sheet_name,
      cols = 1:ncol(data),
      widths = "auto"
    )
  }
}


# --------------------------------------------------------------
# Export standardized source data
# --------------------------------------------------------------

write_formatted_sheet(
  workbook,
  "Standardized_Data",
  as.data.frame(all_data)
)


# --------------------------------------------------------------
# Export ANOVA results
# --------------------------------------------------------------

write_formatted_sheet(
  workbook,
  "ANOVA_TypeIII",
  anova_table
)

write_formatted_sheet(
  workbook,
  "ANOVA_Compact",
  compact_anova
)

write_formatted_sheet(
  workbook,
  "Interpretation_Guide",
  interaction_decisions
)


# --------------------------------------------------------------
# Export main effects
# --------------------------------------------------------------

write_formatted_sheet(
  workbook,
  "MainEffect_Tillage",
  tillage_table
)

write_formatted_sheet(
  workbook,
  "MainEffect_Rotation",
  rotation_table
)


# --------------------------------------------------------------
# Export interaction and simple effects
# --------------------------------------------------------------

write_formatted_sheet(
  workbook,
  "Interaction_EMMeans",
  interaction_table
)

write_formatted_sheet(
  workbook,
  "Tillage_within_Rotation",
  tillage_within_rotation_table
)

write_formatted_sheet(
  workbook,
  "Rotation_within_Tillage",
  rotation_within_tillage_table
)


# --------------------------------------------------------------
# Export diagnostics and analysis logs
# --------------------------------------------------------------

write_formatted_sheet(
  workbook,
  "Model_Diagnostics",
  diagnostics_table
)

write_formatted_sheet(
  workbook,
  "Analysis_Log",
  analysis_log_table
)

write_formatted_sheet(
  workbook,
  "MAOM_CN_75_100_Check",
  maom_cn_check
)


# --------------------------------------------------------------
# Save the workbook
# --------------------------------------------------------------

saveWorkbook(
  workbook,
  output_file,
  overwrite = TRUE
)

cat(
  "\nAnalysis workbook successfully created:\n",
  output_file,
  "\n"
)


Analysis workbook successfully created:
 /content/Complete_Split_Plot_Analysis.xlsx 


In [58]:
%%R # ==============================================================
# STEP 17: DOWNLOAD THE RESULTS WORKBOOK
# ==============================================================

system(
  paste(
    "ls -lh",
    shQuote(output_file)
  )
)

cat(
  "\nThe workbook is available in the Colab Files panel:\n",
  output_file,
  "\n"
)


The workbook is available in the Colab Files panel:
 /content/Complete_Split_Plot_Analysis.xlsx 
